# FinGuard Fraud Detection Pipeline
## Notebook 01 — Bronze Layer Ingestion

---

| | |
|---|---|
| **Layer** | Bronze (Raw Ingestion) |
| **Source** | `/Volumes/finguard/raw/source_files/` |
| **Target** | `finguard.bronze.*` (Delta Lake) |
| **Pattern** | Medallion Architecture — Bronze Layer |
| **Context** | Australian retail banking (CBA / NAB / Westpac style) |

---

### What This Notebook Does

The Bronze layer is the **raw landing zone**. The rule is simple:  
**land data exactly as received, enforce schema, add audit columns — no business logic.**

| Step | Description |
|------|-------------|
| 1 | Configure Unity Catalog paths and Spark session |
| 2 | Define explicit StructType schemas |
| 3 | Ingest CSVs from Unity Catalog Volume |
| 4 | Add audit columns (ingestion timestamp, source, batch ID) |
| 5 | Write to Delta Lake tables in `finguard.bronze` |
| 6 | Run data quality checks |
| 7 | Explore Delta Lake features (history, time travel, OPTIMIZE, VACUUM) |

## Cell 1 — Imports

In [0]:
%sql
DROP TABLE IF EXISTS finguard.bronze.transactions;
DROP TABLE IF EXISTS finguard.bronze.customers;
DROP TABLE IF EXISTS finguard.bronze.merchants;

In [0]:
raw_csv = spark.read.option("header", "true").csv(
    "/Volumes/finguard/raw/source_files/transactions.csv"
)
print(raw_csv.columns)

In [0]:
# ── Standard library ──────────────────────────────────────────────────────────
from datetime import datetime

# ── PySpark ───────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# ── Delta Lake ────────────────────────────────────────────────────────────────
# Built into Databricks — no pip install needed
from delta.tables import DeltaTable

# ── spark and dbutils are pre-injected by Databricks serverless ───────────────

print(f"Spark version   : {spark.version}")
print(f"Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} AEST")

## Cell 2 — Configuration

> **Interview point:** Always parameterise paths — never hardcode in notebook cells.  
> In production these come from a config YAML, Databricks widgets, or a secrets vault.

In [0]:
# ── Unity Catalog identifiers ─────────────────────────────────────────────────
CATALOG_NAME = "finguard"
RAW_SCHEMA   = "raw"
BRONZE_SCHEMA = "bronze"

# ── Source paths (Unity Catalog Volume) ───────────────────────────────────────
VOLUME_BASE           = f"/Volumes/{CATALOG_NAME}/{RAW_SCHEMA}/source_files"
SOURCE_TRANSACTIONS   = f"{VOLUME_BASE}/transactions.csv"
SOURCE_CUSTOMERS      = f"{VOLUME_BASE}/customers.csv"
SOURCE_MERCHANTS      = f"{VOLUME_BASE}/merchants.csv"

# ── Target table names (Unity Catalog: catalog.schema.table) ──────────────────
BRONZE_TRANSACTIONS   = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.transactions"
BRONZE_CUSTOMERS      = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.customers"
BRONZE_MERCHANTS      = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.merchants"

# ── Batch ID — uniquely identifies this ingestion run (data lineage) ──────────
BATCH_ID = f"bronze_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PIPELINE_NAME = "finguard_bronze_ingestion"

print(f"Catalog       : {CATALOG_NAME}")
print(f"Volume base   : {VOLUME_BASE}")
print(f"Bronze schema : {CATALOG_NAME}.{BRONZE_SCHEMA}")
print(f"Batch ID      : {BATCH_ID}")

In [0]:
# ── Spark configuration ───────────────────────────────────────────────────────
# Databricks Serverless manages ALL Spark and Delta configurations automatically.
# spark.conf.set is not available on serverless — the platform optimises everything
# internally including AQE, autoMerge, optimizeWrite, and autoCompact.

print("✓ Running on Databricks Serverless")
print(f"  Spark version : {spark.version}")
print("  AQE, Delta optimisations — managed automatically by platform")

## Cell 3 — Create Bronze Schema

> **Interview point:** In Unity Catalog, `CREATE SCHEMA IF NOT EXISTS` is the standard way  
> to create a namespace. The 3-level hierarchy is: `catalog.schema.table`.

In [0]:
# Create the bronze schema if it doesn't already exist
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}
    COMMENT 'FinGuard Bronze layer — raw ingestion from payment gateway feeds'
""")

# Set as default so subsequent SQL cells don't need the full qualifier
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

print(f"✓ Schema ready: {CATALOG_NAME}.{BRONZE_SCHEMA}")

## Cell 4 — Schema Definitions

> **Interview point — the most common PySpark interview question:**  
> *'Why define an explicit schema instead of using inferSchema?'*  
> 
> **Answer:** `inferSchema=True` triggers a full extra scan of your entire dataset  
> just to guess column types — expensive on 500k+ rows. It can also guess wrong  
> (e.g. reads `0` and infers `IntegerType` instead of `BooleanType`), creating  
> silent type errors that break downstream Silver transforms.  
> Explicit schemas are deterministic, documented, and fast.

In [0]:
# ── Transaction schema ────────────────────────────────────────────────────────
# Mirrors a payment gateway feed from Visa DPS or eftpos Australia.
# NOTE: txn_timestamp kept as StringType here — cast to TimestampType in Silver.
#       Bronze rule: land raw, don't transform.

TRANSACTION_SCHEMA = StructType([
    StructField("transaction_id",    StringType(),  nullable=False),
    StructField("customer_id",       StringType(),  nullable=False),
    StructField("merchant_id",       StringType(),  nullable=False),
    StructField("txn_timestamp",     StringType(),  nullable=False),
    StructField("txn_date",          StringType(),  nullable=False),
    StructField("txn_hour",          IntegerType(), nullable=True),
    StructField("txn_day_of_week",   StringType(),  nullable=True),
    StructField("amount",            DoubleType(),  nullable=False),
    StructField("currency",          StringType(),  nullable=True),
    StructField("channel",           StringType(),  nullable=True),
    StructField("card_network",      StringType(),  nullable=True),
    StructField("merchant_category", StringType(),  nullable=True),
    StructField("mcc_code",          StringType(),  nullable=True),
    StructField("response_code",     StringType(),  nullable=True),
    StructField("is_declined",       BooleanType(), nullable=True),
    StructField("is_international",  BooleanType(), nullable=True),
    StructField("device_type",       StringType(),  nullable=True),
    StructField("ip_country",        StringType(),  nullable=True),
    StructField("is_fraud",          BooleanType(), nullable=True),
    StructField("fraud_type",        StringType(),  nullable=True),
    StructField("fraud_indicator",   StringType(),  nullable=True),
    StructField("partition_date",    StringType(),  nullable=True),
])

print(f"Transaction schema: {len(TRANSACTION_SCHEMA.fields)} fields defined")

In [0]:
# ── Customer schema ───────────────────────────────────────────────────────────

CUSTOMER_SCHEMA = StructType([
    StructField("customer_id",        StringType(),  nullable=False),
    StructField("first_name",         StringType(),  nullable=True),
    StructField("last_name",          StringType(),  nullable=True),
    StructField("date_of_birth",      StringType(),  nullable=True),
    StructField("gender",             StringType(),  nullable=True),
    StructField("email",              StringType(),  nullable=True),
    StructField("phone",              StringType(),  nullable=True),
    StructField("address_street",     StringType(),  nullable=True),
    StructField("address_suburb",     StringType(),  nullable=True),
    StructField("address_state",      StringType(),  nullable=True),
    StructField("address_postcode",   StringType(),  nullable=True),
    StructField("annual_income_aud",  DoubleType(),  nullable=True),
    StructField("employment_status",  StringType(),  nullable=True),
    StructField("credit_score",       IntegerType(), nullable=True),
    StructField("account_open_date",  StringType(),  nullable=True),
    StructField("is_high_risk",       BooleanType(), nullable=True),
    StructField("kyc_verified",       BooleanType(), nullable=True),
])

# ── Merchant schema ───────────────────────────────────────────────────────────

MERCHANT_SCHEMA = StructType([
    StructField("merchant_id",      StringType(),  nullable=False),
    StructField("merchant_name",    StringType(),  nullable=True),
    StructField("category",         StringType(),  nullable=True),
    StructField("mcc_code",         StringType(),  nullable=True),
    StructField("abn",              StringType(),  nullable=True),
    StructField("address_suburb",   StringType(),  nullable=True),
    StructField("address_state",    StringType(),  nullable=True),
    StructField("country",          StringType(),  nullable=True),
    StructField("is_international", BooleanType(), nullable=True),
    StructField("is_online_only",   BooleanType(), nullable=True),
    StructField("risk_level",       StringType(),  nullable=True),
    StructField("registered_date",  StringType(),  nullable=True),
])

print(f"Customer schema : {len(CUSTOMER_SCHEMA.fields)} fields defined")
print(f"Merchant schema : {len(MERCHANT_SCHEMA.fields)} fields defined")

## Cell 5 — Helper Functions

In [0]:
def read_csv_with_schema(path, schema):
    """Read a CSV file with an explicit schema.

    Args:
        path:   Full path to the CSV file (Unity Catalog Volume or DBFS).
        schema: StructType schema to enforce on read.

    Returns:
        Spark DataFrame.

    Interview point — read modes:
        PERMISSIVE   : Malformed rows become null. Safe default — keeps pipeline
                       flowing. Pair with a dead-letter table to capture bad rows.
        FAILFAST     : Throws exception on first bad row. Use for critical feeds
                       where a bad row means the whole file is suspect.
        DROPMALFORMED: Silently drops bad rows. NEVER use in banking — data loss
                       without a trace violates APRA CPS 234.
    """
    return (
        spark.read
        .option("header",           "true")
        .option("mode",             "PERMISSIVE")
        .option("nullValue",        "")
        .option("dateFormat",       "yyyy-MM-dd")
        .option("timestampFormat",  "yyyy-MM-dd HH:mm:ss")
        .schema(schema)
        .csv(path)
    )


def add_audit_columns(df, source_file, batch_id):
    """Add standard audit columns to every Bronze table.

    Interview point:
        Audit columns answer: When was this row loaded? From where? Which batch?
        Non-negotiable in Australian banking under APRA CPS 234 (data integrity)
        and AUSTRAC AML/CTF Act (transaction record-keeping obligations).
    """
    return (
        df
        .withColumn("_ingested_at",   F.current_timestamp())
        .withColumn("_source_file",   F.lit(source_file))
        .withColumn("_batch_id",      F.lit(batch_id))
        .withColumn("_pipeline_name", F.lit(PIPELINE_NAME))
    )


def write_bronze_table(df, table_name, partition_col=None):
    """Write a DataFrame to a Unity Catalog Delta table.

    Args:
        df:            Spark DataFrame to write.
        table_name:    Fully qualified table name (catalog.schema.table).
        partition_col: Optional column to partition by.

    Interview point — why Delta over plain Parquet?
        1. ACID transactions  — no corrupt reads during concurrent writes
        2. Time travel        — query any historical version
        3. Schema enforcement — rejects schema-breaking writes
        4. DML support        — UPDATE, DELETE, MERGE (impossible in Parquet)
        5. Transaction log    — full audit trail in _delta_log/
    """
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(table_name)


print("✓ Helper functions defined")

## Cell 6 — Ingest Customers

In [0]:
print("Ingesting customers...")

customers_raw    = read_csv_with_schema(SOURCE_CUSTOMERS, CUSTOMER_SCHEMA)
customers_bronze = add_audit_columns(customers_raw, SOURCE_CUSTOMERS, BATCH_ID)

# Partitioned by address_state — common in AU banking for state-based reporting
write_bronze_table(customers_bronze, BRONZE_CUSTOMERS, partition_col="address_state")

count = spark.table(BRONZE_CUSTOMERS).count()
print(f"✓ {BRONZE_CUSTOMERS}: {count:,} rows written")

In [0]:
# Preview
display(spark.table(BRONZE_CUSTOMERS).limit(5))

## Cell 7 — Ingest Merchants

In [0]:
print("Ingesting merchants...")

merchants_raw    = read_csv_with_schema(SOURCE_MERCHANTS, MERCHANT_SCHEMA)
merchants_bronze = add_audit_columns(merchants_raw, SOURCE_MERCHANTS, BATCH_ID)

# No partition — merchants is a small lookup table (~800 rows)
write_bronze_table(merchants_bronze, BRONZE_MERCHANTS)

count = spark.table(BRONZE_MERCHANTS).count()
print(f"✓ {BRONZE_MERCHANTS}: {count:,} rows written")

## Cell 8 — Ingest Transactions (the big table)

In [0]:
print("Ingesting transactions (500,000 rows)...")

txn_raw = read_csv_with_schema(SOURCE_TRANSACTIONS, TRANSACTION_SCHEMA)

# ── Repartition before writing ─────────────────────────────────────────────
# Interview point: Why repartition?
# Spark default is spark.sql.shuffle.partitions = 200.
# For 500k rows that creates 200 tiny files — the small file problem.
# Repartition by partition_date aligns partitions with our physical partition layout,
# so each partition_month folder gets evenly sized files.
txn_raw = txn_raw.repartition(12, "partition_date")  # 12 months of data

txn_bronze = add_audit_columns(txn_raw, SOURCE_TRANSACTIONS, BATCH_ID)

# Partitioned by partition_date — queries for a date range only scan relevant folders
# (partition pruning). Standard in AU banking data lakes.
write_bronze_table(txn_bronze, BRONZE_TRANSACTIONS, partition_col="partition_date")

count = spark.table(BRONZE_TRANSACTIONS).count()
print(f"✓ {BRONZE_TRANSACTIONS}: {count:,} rows written")

In [0]:
for row in spark.table("finguard.bronze.transactions").limit(1).collect():
    print(f"amount        : {row['amount']}")
    print(f"currency      : {row['currency']}")
    print(f"channel       : {row['channel']}")

In [0]:
# Schema confirmation
spark.table(BRONZE_TRANSACTIONS).printSchema()

## Cell 9 — Data Quality Checks

> **Australian banking context:**  
> Data quality is regulated under **APRA CPS 234** (data integrity) and  
> **AUSTRAC AML/CTF Act** (transaction reporting accuracy).  
> Every Bronze table must pass DQ checks before Silver processing begins.

In [0]:
def run_dq_report(df, table_name):
    """Run data quality checks and print a summary report.

    In production these results write to a finguard.monitoring.dq_results
    Delta table so you can trend data quality over time.
    """
    total = df.count()
    print(f"\n{'─' * 55}")
    print(f"  DQ Report: {table_name}")
    print(f"  Total rows: {total:,}")
    print(f"{'─' * 55}")

    # Null counts for non-audit columns
    business_cols = [c for c in df.columns if not c.startswith("_")]
    null_counts = (
        df.select([
            F.count(F.when(F.col(c).isNull(), c)).alias(c)
            for c in business_cols
        ])
        .collect()[0]
        .asDict()
    )

    has_nulls = False
    for col_name, null_count in null_counts.items():
        if null_count > 0:
            pct  = null_count / total * 100
            flag = " ⚠️  REVIEW" if pct > 5 else ""
            print(f"  {col_name:<35} {null_count:>8,}  ({pct:5.1f}%){flag}")
            has_nulls = True

    if not has_nulls:
        print("  No nulls found in any column ✓")


run_dq_report(spark.table(BRONZE_TRANSACTIONS), BRONZE_TRANSACTIONS)
run_dq_report(spark.table(BRONZE_CUSTOMERS),    BRONZE_CUSTOMERS)
run_dq_report(spark.table(BRONZE_MERCHANTS),    BRONZE_MERCHANTS)

In [0]:
# ── Transaction-specific DQ ────────────────────────────────────────────────
txn_df = spark.table(BRONZE_TRANSACTIONS)
total  = txn_df.count()

print("Transaction-specific quality checks")
print("─" * 45)

# 1. Negative or zero amounts
bad_amounts = txn_df.filter(F.col("amount").cast(DoubleType()) <= 0).count()
print(f"  Negative / zero amounts    : {bad_amounts:,}")

# 2. Duplicate transaction IDs — payment gateways must be idempotent
total_ids    = txn_df.count()
distinct_ids = txn_df.select("transaction_id").distinct().count()
duplicates   = total_ids - distinct_ids
print(f"  Duplicate transaction IDs  : {duplicates:,}")

# 3. Currency check — expect AUD only for this domestic feed
print("\n  Currency distribution:")
display(
    txn_df.groupBy("currency")
    .count()
    .orderBy(F.desc("count"))
)

# 4. Fraud rate sanity check
fraud_rate = txn_df.filter(F.col("is_fraud") == True).count() / total * 100
print(f"  Fraud rate                 : {fraud_rate:.2f}%  (expected ~3-5%)")

# 5. AUSTRAC structuring check — transactions just under $10k threshold
structuring_count = txn_df.filter(
    F.col("amount").between(9_000, 9_999)
).count()
print(f"  Near-threshold ($9k-$10k)  : {structuring_count:,}  (AUSTRAC flag)")

In [0]:
spark.table("finguard.bronze.transactions").printSchema()
for row in spark.table("finguard.bronze.transactions").limit(1).collect():
    print(row.asDict())

## Cell 10 — Delta Lake Features

These are the features interviewers ask about most. Every cell below has the answer embedded.

In [0]:
# ── 10a. TABLE HISTORY ────────────────────────────────────────────────────────
# Interview point:
# DESCRIBE HISTORY shows every operation ever run on a Delta table —
# who ran it, when, what operation, what files changed.
# Essential for debugging, auditing, and regulatory compliance.

print("Table history (full audit log):")
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TRANSACTIONS}"))

In [0]:
# ── 10b. TIME TRAVEL ──────────────────────────────────────────────────────────
# Interview point:
# Delta stores every version of a table via its transaction log (_delta_log/).
# You can query any previous version using versionAsOf or timestampAsOf.
# Versions are retained until VACUUM is run (default retention: 7 days).
#
# Real use cases in AU banking:
#   - Auditor asks: what did this table look like on 30 June 2024?
#   - A bad pipeline run corrupted data — revert to the last good version.
#   - Regulatory reporting requires point-in-time data snapshots.

df_version_0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .table(BRONZE_TRANSACTIONS)
)
print(f"Version 0 row count: {df_version_0.count():,}")

# Alternatively by timestamp:
# spark.read.format("delta").option("timestampAsOf", "2024-01-01").table(BRONZE_TRANSACTIONS)

In [0]:
# ── 10c. OPTIMIZE and ZORDER ──────────────────────────────────────────────────
# Interview point:
# OPTIMIZE  : Compacts many small Parquet files into fewer larger files.
#             Solves the small file problem. Schedule daily after ingestion.
# ZORDER BY : Co-locates related rows in the same files (multi-dimensional index).
#             Queries filtering on ZORDERed columns skip far more files.
#
# Why ZORDER by customer_id and txn_date?
# Fraud investigation queries are always:
#   "show me all transactions for customer X in date range Y"
# ZORDERing on those columns means Spark reads the minimum possible files.

print("Running OPTIMIZE + ZORDER on bronze transactions...")
print("(Schedule this daily in production — after each ingestion run)")

spark.sql(f"""
    OPTIMIZE {BRONZE_TRANSACTIONS}
    ZORDER BY (customer_id, txn_date)
""")

print("✓ OPTIMIZE complete")

In [0]:
# ── 10d. VACUUM ───────────────────────────────────────────────────────────────
# Interview point:
# VACUUM removes old data files no longer referenced by the transaction log.
# Default retention: 168 hours (7 days) — preserves 7 days of time travel.
# DO NOT set retentionHours=0 in production — destroys all time travel history.
#
# DRY RUN shows what would be deleted without actually deleting anything.

print("VACUUM dry run — showing files that would be removed:")
display(spark.sql(f"VACUUM {BRONZE_TRANSACTIONS} RETAIN 168 HOURS DRY RUN"))

# To actually run in production:
# spark.sql(f"VACUUM {BRONZE_TRANSACTIONS} RETAIN 168 HOURS")

## Cell 11 — Summary

In [0]:
print("═" * 60)
print("  BRONZE LAYER COMPLETE")
print("═" * 60)
print(f"  Batch ID  : {BATCH_ID}")
print()

bronze_tables = [
    BRONZE_TRANSACTIONS,
    BRONZE_CUSTOMERS,
    BRONZE_MERCHANTS,
]

for table in bronze_tables:
    count = spark.table(table).count()
    print(f"  {table:<45} {count:>10,} rows")

print()
print("  Delta features applied:")
print("    ✓ Explicit StructType schema enforcement")
print("    ✓ PERMISSIVE ingestion with audit columns")
print("    ✓ Unity Catalog Delta tables")
print("    ✓ Partition by txn_month (transaction pruning)")
print("    ✓ Data quality checks (APRA / AUSTRAC aligned)")
print("    ✓ OPTIMIZE + ZORDER BY (customer_id, txn_date)")
print("    ✓ Time travel (version 0 verified)")
print("    ✓ VACUUM dry run")
print()
print("  Next → notebooks/02_silver_transformation.ipynb")
print("═" * 60)

---
## Interview Question Reference

| Question | Cell |
|----------|------|
| Why explicit schema over inferSchema? | Cell 4 |
| What is the Medallion Architecture? | Cell 1 intro |
| What are PERMISSIVE / FAILFAST / DROPMALFORMED? | Cell 5 |
| What is Delta Lake and why use it over Parquet? | Cell 5 |
| What are audit columns and why do they matter? | Cell 5 |
| What is AQE (Adaptive Query Execution)? | Cell 2 |
| What is the small file problem? | Cell 2, Cell 8 |
| What is repartition and when should you use it? | Cell 8 |
| What is partition pruning? | Cell 8 |
| What is DESCRIBE HISTORY? | Cell 10a |
| What is Delta Time Travel? | Cell 10b |
| What is OPTIMIZE and ZORDER? | Cell 10c |
| What is VACUUM? | Cell 10d |
| What is Unity Catalog and the 3-level namespace? | Cell 2 |
| What is APRA CPS 234? | Cell 9 |
| What is AUSTRAC and the $10k threshold? | Cell 9 |

In [0]:
# Quick verification
tables = [
    "finguard.bronze.transactions",
    "finguard.bronze.customers",
    "finguard.bronze.merchants",
]

for table in tables:
    count = spark.table(table).count()
    print(f"  ✓ {table:<45} {count:,} rows")